In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 88 — Claims Processing Assistant

## Goal
Build a claims processing pipeline that **intakes forms**,
**extracts fields**, **summarizes the claim**, and **routes
edge cases** to human review.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Field extraction** | LLM extracts structured data from text |
| **Validation rules** | Business logic for auto-approve vs flag |
| **Conditional routing** | LangGraph edges based on claim type |
| **Human-in-the-loop** | Flag complex claims for manual review |

## Stack
- `langgraph` — stateful workflow with conditional routing
- `langchain-ollama` — LLM for extraction and summarization
- Ollama `qwen3.5:9b`

> **Note:** OCR is simulated — we use text-based claim forms.
> In production, use PaddleOCR or Tesseract for image processing.

In [ ]:
import os, warnings, json
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOllama(model="qwen3.5:9b", temperature=0)
print("All imports OK")

## Step 1 — Sample Claim Forms (Simulated OCR Output)

In [ ]:
# Simulated claim forms (as if OCR already extracted text)
CLAIM_FORMS = [
    {
        "claim_id": "CLM-2024-001",
        "raw_text": """INSURANCE CLAIM FORM
Date: November 5, 2024
Policy #: POL-AUTO-78234
Claimant: John Martinez
Type: Auto / Collision
Date of Loss: October 28, 2024
Description: Rear-ended at traffic light on Main St. Other driver
admitted fault. Police report #PR-2024-5567 filed.
Damage: Bumper, trunk, rear lights. Vehicle driveable.
Estimated Repair: $3,200
Injuries: None reported
Third Party: Yes — other driver's insurance is SafeGuard (claim pending)"""
    },
    {
        "claim_id": "CLM-2024-002",
        "raw_text": """INSURANCE CLAIM FORM
Date: November 8, 2024
Policy #: POL-HOME-45123
Claimant: Sarah Williams
Type: Homeowner / Water Damage
Date of Loss: November 1, 2024
Description: Pipe burst in upstairs bathroom causing water damage
to bathroom floor, ceiling of kitchen below, and kitchen cabinets.
Damage was discovered after returning from 3-day trip. Plumber
states pipe was corroded.
Estimated Repair: $28,500
Temporary Housing: Requesting $4,000 for 2 weeks hotel
Prior Claims: Water damage claim in 2022 ($12,000)"""
    },
    {
        "claim_id": "CLM-2024-003",
        "raw_text": """INSURANCE CLAIM FORM
Date: November 10, 2024
Policy #: POL-AUTO-91056
Claimant: David Chen
Type: Auto / Theft
Date of Loss: November 9, 2024
Description: Vehicle stolen from parking garage at 555 Oak Ave.
No security camera footage available (cameras were offline per
garage management). No witnesses. Vehicle is a 2023 BMW X5,
estimated value $62,000. Second auto theft claim in 18 months.
Police report filed: PR-2024-8821.
Injuries: N/A"""
    },
]

print(f"Loaded {len(CLAIM_FORMS)} claim forms")

## Step 2 — LangGraph Claims Workflow

```
extract_fields → validate_claim → [simple?]
    → YES: auto_process → summarize → END
    → NO:  flag_for_review → summarize → END
```

In [ ]:
class ClaimState(TypedDict):
    claim_id: str
    raw_text: str
    extracted_fields: dict
    validation_result: dict
    route: str
    processing_notes: list
    summary: str

# Node 1: Extract structured fields from claim text
def extract_fields(state: ClaimState) -> ClaimState:
    print(f"\n📋 Extracting fields from {state['claim_id']}...")
    
    msg = llm.invoke([
        SystemMessage(content="""Extract structured data from this insurance claim form.
Return a JSON object with keys:
- policy_number, claimant_name, claim_type, date_of_loss
- description (brief), estimated_amount (number), injuries (yes/no)
- third_party (yes/no), prior_claims (yes/no), prior_claims_detail
- red_flags (list of any suspicious indicators)

Return ONLY the JSON object."""),
        HumanMessage(content=state["raw_text"])
    ])
    
    raw = msg.content
    if "<think>" in raw:
        raw = raw.split("</think>")[-1].strip()
    if "```" in raw:
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()
    
    try:
        start = raw.find("{")
        end = raw.rfind("}") + 1
        fields = json.loads(raw[start:end])
    except (json.JSONDecodeError, ValueError):
        fields = {"error": "extraction_failed", "raw": raw[:200]}
    
    print(f"  Type: {fields.get('claim_type', '?')}")
    print(f"  Amount: ${fields.get('estimated_amount', '?')}")
    print(f"  Red flags: {fields.get('red_flags', [])}")
    
    return {**state, "extracted_fields": fields}

print("Node: extract_fields defined")

In [ ]:
# Node 2: Validate claim against business rules
def validate_claim(state: ClaimState) -> ClaimState:
    print(f"  🔍 Validating {state['claim_id']}...")
    
    fields = state["extracted_fields"]
    flags = []
    
    # Business rules
    amount = fields.get("estimated_amount", 0)
    if isinstance(amount, str):
        amount = float(amount.replace(",", "").replace("$", "")) if amount else 0
    
    if amount > 25000:
        flags.append(f"HIGH_VALUE: ${amount:,.0f} exceeds $25K auto-approve threshold")
    
    if fields.get("prior_claims") in ["yes", True, "Yes"]:
        flags.append("REPEAT_CLAIMANT: Prior claims on record")
    
    red_flags = fields.get("red_flags", [])
    if red_flags:
        flags.extend([f"RED_FLAG: {rf}" for rf in red_flags])
    
    claim_type = str(fields.get("claim_type", "")).lower()
    if "theft" in claim_type:
        flags.append("THEFT_CLAIM: Requires SIU review")
    
    route = "auto_process" if len(flags) == 0 else "flag_for_review"
    
    print(f"  Flags: {len(flags)}")
    for f in flags:
        print(f"    ⚠️ {f}")
    print(f"  Route: {route.upper()}")
    
    return {**state, "validation_result": {"flags": flags, "flag_count": len(flags)}, "route": route}

# Router function
def route_claim(state: ClaimState) -> str:
    return state["route"]

print("Node: validate_claim defined")

In [ ]:
# Node 3a: Auto-process (simple claims)
def auto_process(state: ClaimState) -> ClaimState:
    print(f"  ✅ Auto-processing {state['claim_id']}...")
    notes = state.get("processing_notes", []) + [
        "DECISION: Auto-approved",
        "No flags detected",
        f"Estimated payout: ${state['extracted_fields'].get('estimated_amount', 'TBD')}"
    ]
    return {**state, "processing_notes": notes}

# Node 3b: Flag for human review
def flag_for_review(state: ClaimState) -> ClaimState:
    print(f"  🚩 Flagging {state['claim_id']} for human review...")
    flags = state["validation_result"]["flags"]
    notes = state.get("processing_notes", []) + [
        "DECISION: Routed to human review",
        f"Reason: {len(flags)} flag(s) detected",
    ] + [f"FLAG: {f}" for f in flags]
    return {**state, "processing_notes": notes}

# Node 4: Generate summary
def summarize_claim(state: ClaimState) -> ClaimState:
    print(f"  📝 Generating summary for {state['claim_id']}...")
    
    context = json.dumps({
        "claim_id": state["claim_id"],
        "extracted_fields": state["extracted_fields"],
        "validation_flags": state["validation_result"]["flags"],
        "processing_notes": state["processing_notes"]
    }, indent=2, default=str)
    
    msg = llm.invoke([
        SystemMessage(content="""Write a brief claim summary (3-5 sentences) suitable for
an adjuster's review queue. Include: claim type, amount, key facts, and
any flags or concerns. Be factual and concise. No thinking."""),
        HumanMessage(content=context)
    ])
    
    summary = msg.content
    if "<think>" in summary:
        summary = summary.split("</think>")[-1].strip()
    
    return {**state, "summary": summary}

print("Nodes 3a, 3b, 4 defined")

In [ ]:
# Build the graph
graph = StateGraph(ClaimState)

graph.add_node("extract", extract_fields)
graph.add_node("validate", validate_claim)
graph.add_node("auto_process", auto_process)
graph.add_node("flag_for_review", flag_for_review)
graph.add_node("summarize", summarize_claim)

graph.set_entry_point("extract")
graph.add_edge("extract", "validate")
graph.add_conditional_edges("validate", route_claim, {
    "auto_process": "auto_process",
    "flag_for_review": "flag_for_review"
})
graph.add_edge("auto_process", "summarize")
graph.add_edge("flag_for_review", "summarize")
graph.add_edge("summarize", END)

claims_app = graph.compile()
print("Claims processing graph compiled!")

In [ ]:
# Process all claims
results = []
for claim in CLAIM_FORMS:
    print("\n" + "=" * 60)
    print(f"PROCESSING: {claim['claim_id']}")
    print("=" * 60)
    
    result = claims_app.invoke({
        "claim_id": claim["claim_id"],
        "raw_text": claim["raw_text"],
        "extracted_fields": {},
        "validation_result": {},
        "route": "",
        "processing_notes": [],
        "summary": ""
    })
    results.append(result)

# Print dashboard
print("\n\n" + "=" * 60)
print("CLAIMS DASHBOARD")
print("=" * 60)
for r in results:
    route = r["route"].upper()
    icon = "✅" if route == "AUTO_PROCESS" else "🚩"
    print(f"\n{icon} {r['claim_id']} — {route}")
    print(f"   {r['summary'][:200]}")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Field extraction** | LLM parses unstructured claim text → JSON |
| **Business rules** | Amount threshold, repeat claims, theft flag |
| **Conditional routing** | `add_conditional_edges` for auto vs review |
| **Claim summary** | LLM generates adjuster-ready summary |

## Architecture
```
Claim Form (text/OCR) → Extract Fields (LLM)
            ↓
     Validate (business rules)
     ┌────────┴────────┐
     ↓ No flags        ↓ Flags
  Auto-Process    Flag for Review
     └─────┬──────────┘
           ↓
     Summarize (LLM)
           ↓
          END
```

## Extending This Project
- Add PaddleOCR for real document scanning
- Fraud scoring model (ML-based)
- Integration with claim management systems
- Multi-document claims (photos, receipts, police reports)